将 Query、Key、Value 通过独立线性层投影到多个子空间，在每个子空间独立执行缩放点积注意力计算，最后将所有头的输出拼接融合，让模型同时捕捉不同维度的语义特征，表达能力高于单头注意力

In [1]:
import math
import torch
from torch import nn

def transpose_qkv(X, num_heads):
    "点积注意力本身支持批量计算，把 batch × heads 合并成一个维度，从而能一次性跑完所有头的注意力"
    # 输入X形状: (batch_size, 序列长度, num_hiddens)

    # 步骤1：把最后一维隐藏层拆成「头数 × 单头维度」
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    # 此时形状: (batch_size, seq_len, num_heads, head_dim)
    
    # 步骤2：交换维度，把头维度放到batch之后，方便后续合并
    X = X.permute(0, 2, 1, 3)
    # 此时形状: (batch_size, num_heads, seq_len, head_dim)
    
    # 步骤3：合并batch与num_heads，伪装成更大的batch
    return X.reshape(-1, X.shape[2], X.shape[3]) # 输出形状: (batch_size * num_heads, seq_len, head_dim)

def transpose_output(X, num_heads):
    "transpose_qkv的反向操作，把多头注意力的输出还原回原始形状，拼接所有头的结果"
    # 输入X形状: (batch_size * num_heads, seq_len, head_dim)
    
    # 步骤1：拆开batch和num_heads
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    # 此时形状: (batch_size, num_heads, seq_len, head_dim)
    
    # 步骤2：把序列长度换回第二维
    X = X.permute(0, 2, 1, 3)
    # 此时形状: (batch_size, seq_len, num_heads, head_dim)
    
    # 步骤3：合并所有头的维度，还原成完整隐藏层
    return X.reshape(X.shape[0], X.shape[1], -1)
    # 输出形状: (batch_size, seq_len, num_hiddens)

In [2]:
class DotProductAttention(nn.Module):
    """缩放点积注意力
    输入形状:
        queries: (batch_size, n_queries, d_k)
        keys:    (batch_size, n_keys, d_k)
        values:  (batch_size, n_keys, d_v)
        valid_lens: (batch_size,) 或 (batch_size, n_queries)，标记有效 key 的长度
    输出形状:
        (batch_size, n_queries, d_v)
    """

    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]  # 获取 Q/K 公共维度 d

        # 1. 批量点积打分 + 缩放
        # keys.transpose(1,2): [B, n_k, d] → [B, d, n_k]
        # bmm(Q, K.T) = [B, n_q, d] @ [B, d, n_k] = [B, n_q, n_k]
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
    
        # 2. 掩码softmax得到归一化注意力权重
        self.attention_weights = masked_softmax(scores, valid_lens)
        
        # 3. dropout + 加权汇聚Value
        return torch.bmm(self.dropout(self.attention_weights), values)

In [6]:
def sequence_mask(x: torch.Tensor, valid_lens: torch.Tensor, mask_value: float = -1e6) -> torch.Tensor:
    """
    对二维张量每行做尾部掩码：索引 >= valid_len 全部填充 mask_value
    Args:
        x: shape [N, L]，N行序列，每行长度L
        valid_lens: shape [N]，每行对应的有效长度
        mask_value: 掩码填充极小值
    Returns:
        掩码后的同shape张量
    """
    n_seq, seq_len = x.shape
    # 生成位置索引 0,1,...,seq_len-1
    pos = torch.arange(seq_len, device=x.device, dtype=torch.long)
    # 广播构造掩码：[N, L]，True代表需要掩码（超出有效长度）
    mask = pos.unsqueeze(0) >= valid_lens.unsqueeze(1)
    # 原地替换掩码区域
    x = x.masked_fill(mask, mask_value)
    return x


def masked_softmax(X: torch.Tensor, valid_lens: torch.Tensor = None) -> torch.Tensor:
    """
    带有效长度掩码的Softmax，屏蔽padding无效位置
    Args:
        X: 打分张量 shape [batch, n_query, n_key]
        valid_lens:
            - None: 不做掩码，标准softmax
            - 1D tensor [batch]: 同batch内所有query共享同一个有效长度
            - 2D tensor [batch, n_query]: 每个query独立有效长度
    Returns:
        attention_weights: shape 和输入X一致，每行softmax归一化，掩码位置权重≈0
    """
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)

    batch_size, n_q, n_k = X.shape
    # 统一 valid_lens 到一维 [batch * n_q]，每条query对应一个长度
    if valid_lens.dim() == 1:
        # [B] -> [B * n_q]，每个batch的长度复制n_q份
        valid_lens = torch.repeat_interleave(valid_lens, repeats=n_q)
    elif valid_lens.dim() == 2:
        # [B, n_q] -> [B * n_q]
        valid_lens = valid_lens.reshape(-1)
    else:
        raise ValueError(f"valid_lens仅支持1/2维，当前维度:{valid_lens.dim()}")

    # 保证长度张量和X同设备
    valid_lens = valid_lens.to(X.device)

    # 3D -> 2D [B*n_q, n_k] 批量掩码
    x_2d = X.reshape(-1, n_k)
    x_masked = sequence_mask(x_2d, valid_lens, mask_value=-1e6)

    # 恢复原始三维并softmax
    x_3d = x_masked.reshape(batch_size, n_q, n_k)
    return nn.functional.softmax(x_3d, dim=-1)

In [4]:
class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    支持自注意力、交叉注意力两种场景：
    - 自注意力：queries = keys = values
    - 交叉注意力：queries 来自解码器，keys / values 来自编码器

    Args:
        embed_dim: 最终输出的特征维度（即num_hiddens）
        num_heads: 注意力头数
        dropout: dropout概率
        bias: 线性层是否使用偏置
        kdim: key的输入特征维度，默认等于embed_dim
        vdim: value的输入特征维度，默认等于embed_dim
    """
    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        dropout: float = 0.0,
        bias: bool = False,
        kdim: int = None,
        vdim: int = None,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # 特征维度必须能被头数整除
        if self.head_dim * num_heads != embed_dim:
            raise ValueError(
                f"embed_dim ({embed_dim}) 必须能被 num_heads ({num_heads}) 整除"
            )

        # 默认K/V维度和Q一致
        self.kdim = kdim if kdim is not None else embed_dim
        self.vdim = vdim if vdim is not None else embed_dim

        # 单头缩放点积注意力（复用你的实现）
        self.attention = DotProductAttention(dropout)

        # Q/K/V 投影线性层
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.W_k = nn.Linear(self.kdim, embed_dim, bias=bias)
        self.W_v = nn.Linear(self.vdim, embed_dim, bias=bias)

        # 输出融合层
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=bias)

    def _transpose_qkv(self, x: torch.Tensor) -> torch.Tensor:
        """拆分多头：[B, seq_len, embed_dim] → [B*num_heads, seq_len, head_dim]"""
        batch_size, seq_len, _ = x.shape
        # 拆分特征维度
        x = x.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        # 置换维度：头维度前置
        x = x.permute(0, 2, 1, 3)
        # 合并batch和头维度
        return x.reshape(batch_size * self.num_heads, seq_len, self.head_dim)

    def _transpose_output(self, x: torch.Tensor, batch_size: int) -> torch.Tensor:
        """合并多头：[B*num_heads, seq_len, head_dim] → [B, seq_len, embed_dim]"""
        seq_len = x.shape[1]
        # 拆开batch和头维度
        x = x.reshape(batch_size, self.num_heads, seq_len, self.head_dim)
        # 置换维度：序列维度前置
        x = x.permute(0, 2, 1, 3)
        # 拼接所有头的特征
        return x.reshape(batch_size, seq_len, self.embed_dim)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        valid_lens: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Args:
            query: 查询张量，形状 [batch_size, q_len, embed_dim]
            key:   键张量，形状 [batch_size, k_len, kdim]
            value: 值张量，形状 [batch_size, k_len, vdim]
            valid_lens: 有效长度，形状 [batch_size] 或 [batch_size, q_len]
        Returns:
            多头注意力输出，形状 [batch_size, q_len, embed_dim]
        """
        batch_size = query.shape[0]

        # 1. 线性投影 + 拆分多头
        q = self._transpose_qkv(self.W_q(query))
        k = self._transpose_qkv(self.W_k(key))
        v = self._transpose_qkv(self.W_v(value))

        # 2. 有效长度对齐多头：每个头共享相同的掩码
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, self.num_heads, dim=0)

        # 3. 批量计算所有头的点积注意力
        attn_output = self.attention(q, k, v, valid_lens)

        # 4. 合并多头
        output = self._transpose_output(attn_output, batch_size)

        # 5. 输出投影融合
        return self.W_o(output)